# HotFlip: Gradient-Based Adversarial Attacks on Text

This notebook demonstrates **HotFlip** (Ebrahimi et al., 2018, *"HotFlip: White-Box Adversarial Examples for Text Classification"*), the text analogue of `attacks/FGSM.ipynb` and `attacks/PGD.ipynb` in the companion vision repository. FGSM/PGD compute the gradient of the loss with respect to continuous pixel values and step directly in that direction. Text has no such luxury: a sentence is a sequence of **discrete** tokens, so there is no valid "half a word" to step towards.

HotFlip's solution is to use the gradient as a **local, first-order approximation** of what substituting each token would do to the loss, without actually trying every substitution. It then greedily commits to the single best (position, replacement) pair, exactly the same greedy, gradient-guided spirit as PGD, adapted to a combinatorial search space instead of a continuous one.


In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

### Setup: Target Model
We attack [`distilbert-base-uncased-finetuned-sst-2-english`](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english), a standard, publicly available DistilBERT fine-tuned for binary sentiment classification (SST-2: positive/negative movie reviews), small enough to run this attack on CPU in seconds per sentence.

In [2]:
MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

LABELS = model.config.id2label  # {0: 'NEGATIVE', 1: 'POSITIVE'}
embedding_matrix = model.get_input_embeddings().weight  # (vocab_size, hidden_dim)
print(f"Vocabulary size: {embedding_matrix.shape[0]}, embedding dim: {embedding_matrix.shape[1]}")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

C:\Users\frang\Desktop\adv_at\adversarial_attacks_nlp\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\frang\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Vocabulary size: 30522, embedding dim: 768


In [3]:
def predict(text):
    inputs = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = F.softmax(logits, dim=-1)[0]
    label_idx = int(torch.argmax(probs))
    return LABELS[label_idx], float(probs[label_idx]), label_idx

### The HotFlip Approximation
For a sentence with token embeddings $e_1, \dots, e_n$, HotFlip approximates the change in loss $J$ from replacing token $i$'s embedding $e_i$ with a candidate embedding $e_c$ using a first-order Taylor expansion:
$$\Delta J(i \to c) \approx (e_c - e_i)^\top \nabla_{e_i} J$$
Because this is **linear in $e_c$**, the best substitution across the *entire vocabulary* at position $i$ can be found with a single matrix-vector product against the embedding matrix, no need to run the model once per candidate word, which is exactly what makes this a white-box attack, and exactly what a black-box attack (see `attacks/blackbox/01_SynonymSubstitution.ipynb`) cannot do.


In [4]:
SPECIAL_TOKEN_IDS = set(tokenizer.all_special_ids)

def hotflip_step(input_ids, true_label_idx):
    """Computes, for every non-special position, the best vocabulary substitution under the
    HotFlip linear approximation, and returns the single best (position, candidate_id, score)."""
    embeds = embedding_matrix[input_ids].clone().detach().unsqueeze(0)
    embeds.requires_grad_(True)

    logits = model(inputs_embeds=embeds).logits
    loss = F.cross_entropy(logits, torch.tensor([true_label_idx]))
    loss.backward()

    grad = embeds.grad[0]  # (seq_len, hidden_dim) - gradient of the loss w.r.t. each token embedding

    best_score, best_pos, best_candidate = float('-inf'), None, None
    for pos in range(len(input_ids)):
        if input_ids[pos].item() in SPECIAL_TOKEN_IDS:
            continue
        current_embed = embedding_matrix[input_ids[pos]]
        # (vocab_size,) = (vocab_size, dim) @ (dim,); increasing the loss pushes the model AWAY
        # from true_label_idx, which is exactly what an untargeted attack wants
        scores = (embedding_matrix - current_embed) @ grad[pos]
        candidate_id = int(torch.argmax(scores))
        if scores[candidate_id] > best_score:
            best_score = float(scores[candidate_id])
            best_pos, best_candidate = pos, candidate_id

    return best_pos, best_candidate, best_score

def hotflip_attack(text, max_flips=5):
    """Greedily flips one token at a time until the prediction changes or `max_flips` is reached."""
    input_ids = tokenizer(text, return_tensors='pt')['input_ids'][0]
    _, _, true_label_idx = predict(text)
    flips = []

    for _ in range(max_flips):
        pos, candidate_id, score = hotflip_step(input_ids, true_label_idx)
        original_token = tokenizer.decode([input_ids[pos]])
        new_token = tokenizer.decode([candidate_id])
        input_ids = input_ids.clone()
        input_ids[pos] = candidate_id
        flips.append((original_token, new_token))

        adv_text = tokenizer.decode(input_ids, skip_special_tokens=True)
        _, _, current_label_idx = predict(adv_text)
        if current_label_idx != true_label_idx:
            break

    return tokenizer.decode(input_ids, skip_special_tokens=True), flips

### Evaluation
A handful of clearly-polarized movie review sentences. For each, we report the original
prediction, the sequence of word flips HotFlip chose, and the final (hopefully flipped)
prediction, together with how many words had to change, the text equivalent of the $L_2$
distortion metric used throughout the vision repo.

In [5]:
SAMPLE_REVIEWS = [
    "This movie was absolutely fantastic, I loved every minute of it.",
    "The acting was terrible and the plot made no sense at all.",
    "A brilliant, moving performance that will stay with me for years.",
    "I was bored throughout the entire film and nearly walked out.",
]

for review in SAMPLE_REVIEWS:
    orig_label, orig_prob, _ = predict(review)
    adv_text, flips = hotflip_attack(review, max_flips=5)
    adv_label, adv_prob, _ = predict(adv_text)

    print(f"\n{'='*80}")
    print(f"Original ({orig_label}, {orig_prob*100:.1f}%): {review}")
    print(f"Flips ({len(flips)}): " + ', '.join(f"'{a}'->'{b}'" for a, b in flips))
    print(f"Adversarial ({adv_label}, {adv_prob*100:.1f}%): {adv_text}")
    print(f"Success: {adv_label != orig_label}")

C:\Users\frang\AppData\Local\Temp\ipykernel_25228\2036314267.py:25: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:821.)
  best_score = float(scores[candidate_id])



Original (POSITIVE, 100.0%): This movie was absolutely fantastic, I loved every minute of it.
Flips (2): 'fantastic'->'crap', 'crap'->'failures'
Adversarial (NEGATIVE, 99.7%): this movie was absolutely failures, i loved every minute of it.
Success: True



Original (NEGATIVE, 100.0%): The acting was terrible and the plot made no sense at all.
Flips (3): 'terrible'->'##rdon', 'sense'->'doubt', 'doubt'->'trough'
Adversarial (POSITIVE, 98.6%): the acting wasrdon and the plot made no trough at all.
Success: True



Original (POSITIVE, 100.0%): A brilliant, moving performance that will stay with me for years.
Flips (2): 'brilliant'->'demanding', 'demanding'->'worse'
Adversarial (NEGATIVE, 99.5%): a worse, moving performance that will stay with me for years.
Success: True



Original (NEGATIVE, 100.0%): I was bored throughout the entire film and nearly walked out.
Flips (2): 'bored'->'striking', 'walked'->'arms'
Adversarial (POSITIVE, 100.0%): i was striking throughout the entire film and nearly arms out.
Success: True
